In [ ]:
# One-cell Colab: Runtime -> Change runtime type -> T4 GPU, then run.
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BRANCH = 'feature/hada-im-tta'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/content/tcformer-test')
MNE_DATA = Path('/content/drive/MyDrive/datasets/PhysioNetMI_MNE')
RESULT_ARCHIVE = Path('/content/drive/MyDrive/TCFormer-results/hada_physionet20_results')

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=str(cwd) if cwd else None, env=env, check=True)

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

run(['uv', 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run(['uv', 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run(['uv', 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run([PYTHON, '-c', "import torch; print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(); print('GPU:', torch.cuda.get_device_name(0))"] )

MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
print('===== TRAIN HADA-TCFORMER PHYSIONET FIRST-20 LOSO =====', flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'physionet', '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment)

RESULT_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Train hoàn tất. Kết quả:', archive, flush=True)
